## Extract Olink Protein Matrix (Wide Format)

Produces a **wide-format protein matrix**: rows = `eid × ins_index`, columns = ~2,922 proteins.

### Strategy
Collects data in **chunks by instance + table** to stay under `spark.driver.maxResultSize`,
then assembles and pivots locally with `data.table::dcast()`.


In [ ]:
if (!require(pacman)) install.packages("pacman")
install.packages("sparklyr")
pacman::p_load(sparklyr, data.table, dplyr, stringr, DBI, purrr, bit64, readr)

### 1. Connect to Spark and Identify Database

In [ ]:
port   <- Sys.getenv("SPARK_MASTER_PORT")
master <- paste0("spark://master:", port)

# Increase maxResultSize to 4 GB to give more headroom
config <- spark_config()
config$spark.driver.maxResultSize <- "4G"
config$spark.driver.memory        <- "8G"
sc <- spark_connect(master, config = config)

database_path   <- system("dx find data --class database", intern = TRUE)
app_substring   <- na.omit(str_extract(database_path, '(app\\d+_\\d+)'))
database_substr <- str_extract(
  database_path[str_detect(database_path, app_substring)],
  'database-([A-Za-z0-9]+)'
) %>% tolower() %>% str_replace("database-", "database_")
database <- paste0(database_substr, "__", app_substring)

project_id        <- Sys.getenv('DX_PROJECT_CONTEXT_ID')
record_id         <- system("dx find data --type Dataset --delimiter ',' | awk -F ',' '{print $5}'", intern = TRUE)
project_record_id <- paste0(project_id, ":", record_id)
message("Database: ", database)

### 2. Collect Data in Chunks (One Spark Table at a Time)

Each olink table has ~500 protein columns × ~54k participants — roughly 50-80 MB per collect.
This stays well under the 4 GB limit.

We pivot each chunk to long format **locally** in `data.table`, then `rbindlist` at the end.

In [ ]:
# Define instance layout: instance_id -> number of tables
instances <- list(
  list(inst = 0, n = 12),
  list(inst = 2, n = 6),
  list(inst = 3, n = 6)
)

all_chunks <- list()
chunk_idx  <- 0L
t_total    <- Sys.time()

for (inst_info in instances) {
  inst <- inst_info$inst
  n    <- inst_info$n

  for (i in seq_len(n)) {
    table_name <- sprintf("olink_instance_%d_%04d", inst, i)
    query <- paste0("SELECT * FROM ", database, ".", table_name)

    t0 <- Sys.time()
    message(sprintf("[%d/%d] Collecting %s ...",
                    chunk_idx + 1L, 24L, table_name))

    # Collect this one table (~50-80 MB)
    wide_dt <- sdf_sql(sc, query) %>% collect() %>% as.data.table()

    # Pivot to long: eid | protein_id | result
    protein_cols <- setdiff(names(wide_dt), "eid")
    long_dt <- melt(wide_dt,
                    id.vars       = "eid",
                    measure.vars  = protein_cols,
                    variable.name = "protein_id",
                    value.name    = "result",
                    na.rm         = TRUE)
    long_dt[, ins_index := inst]

    chunk_idx <- chunk_idx + 1L
    all_chunks[[chunk_idx]] <- long_dt

    elapsed <- round(difftime(Sys.time(), t0, units = "secs"), 1)
    message(sprintf("  -> %s rows, %.1fs",
                    format(nrow(long_dt), big.mark = ","), elapsed))

    rm(wide_dt, long_dt); gc(verbose = FALSE)
  }
}

message(sprintf("\nAll 24 tables collected in %.1f minutes.",
                difftime(Sys.time(), t_total, units = "mins")))

In [ ]:
# Combine all chunks into one long data.table
olink_long <- rbindlist(all_chunks, use.names = TRUE)
rm(all_chunks); gc(verbose = FALSE)

message(sprintf("Combined long table: %s rows x %d cols",
                format(nrow(olink_long), big.mark = ","),
                ncol(olink_long)))

# Quick sanity check
message(sprintf("Unique eids: %s | Unique proteins: %s | Instances: %s",
                format(uniqueN(olink_long$eid), big.mark = ","),
                uniqueN(olink_long$protein_id),
                paste(sort(unique(olink_long$ins_index)), collapse = ", ")))

### 3. Pivot to Wide Protein Matrix

In [ ]:
message("Pivoting to wide format with data.table::dcast ...")
t0 <- Sys.time()

protein_matrix <- dcast(olink_long,
                        eid + ins_index ~ protein_id,
                        value.var    = "result",
                        fun.aggregate = mean,  # handles rare duplicates
                        fill = NA_real_)

rm(olink_long); gc(verbose = FALSE)

elapsed <- round(difftime(Sys.time(), t0, units = "secs"), 1)
message(sprintf("Done. Matrix: %s rows x %s cols (%.1fs)",
                format(nrow(protein_matrix), big.mark = ","),
                format(ncol(protein_matrix), big.mark = ","),
                elapsed))

# Preview top-left corner
protein_matrix[1:5, 1:8]

### 4. Save Output

In [ ]:
# CSV output
out_csv <- "../data/protein_matrix.csv"
message("Writing CSV ...")
fwrite(protein_matrix, out_csv)
message(sprintf("Saved %s (%.2f GB)",
                out_csv, file.info(out_csv)$size / 1e9))

In [ ]:
# (Optional) Parquet - much smaller and faster to read back
install.packages("arrow")
library(arrow)
write_parquet(protein_matrix, "../data/protein_matrix.parquet")

### 5. Upload to DNAnexus

In [ ]:
system("dx mkdir -p olink_data")
system("dx upload protein_matrix.csv --path olink_data/protein_matrix.csv")
message("Upload complete.")

In [ ]:
spark_disconnect(sc)
message("Spark disconnected.")